# 3. Inference

This notebook loads the saved bundle in `models/model.joblib`, which contains the fitted preprocessing/model pipeline, the decision threshold, and the expected feature order. I first score a small example directly, then send the same input through FastAPI using its in-process `TestClient`. Comparing both paths checks that the API uses the same model behaviour without needing a separate server or network connection.


In [1]:
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
%matplotlib inline
print("working dir:", Path.cwd())

working dir: C:\Users\Dinesh\Downloads\heart-mlops-submission\heart-mlops-final


In [2]:
import joblib, json
bundle = joblib.load('models/model.joblib')
print({k: (type(v).__name__ if k == 'pipeline' else v) for k, v in bundle.items()})

{'pipeline': 'Pipeline', 'threshold': 0.26, 'features': ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']}


In [3]:
sample = {'age': 57, 'sex': 1, 'cp': 4, 'trestbps': 140, 'chol': 241, 'fbs': 0, 'restecg': 0,
          'thalach': 123, 'exang': 1, 'oldpeak': 0.2, 'slope': 2, 'ca': 0, 'thal': 7}
low = {'age': 35, 'sex': 0, 'cp': 2, 'trestbps': 110, 'chol': 180, 'fbs': 0, 'restecg': 0,
       'thalach': 175, 'exang': 0, 'oldpeak': 0.0, 'slope': 1, 'ca': 0, 'thal': 3}
rows = pd.DataFrame([sample, low])[bundle['features']]
p = bundle['pipeline'].predict_proba(rows)[:, 1]
out = pd.DataFrame({'p_disease': p.round(4), 'prediction': (p >= bundle['threshold']).astype(int)},
                   index=['high-risk sample', 'low-risk sample'])
out

,p_disease,prediction
high-risk sample,0.8480,1
low-risk sample,0.0228,0


## The same examples through the API

I pass the sample payload through the request schema and prediction endpoint. The response includes the predicted class, its label, confidence, and the probability assigned to the disease class. The schema also rejects incomplete or out-of-range inputs instead of silently accepting them.


In [4]:
from fastapi.testclient import TestClient
from src.api import app
with TestClient(app) as client:
    print(client.get('/health').json())
    r1 = client.post('/predict', json=sample); r2 = client.post('/predict', json=low)
    bad = client.post('/predict', json={**sample, 'age': 250})
    print(r1.status_code, r1.json()); print(r2.status_code, r2.json())
    print(bad.status_code, bad.json()['detail'][0]['msg'])
assert abs(r1.json()['probability_disease'] - p[0]) < 1e-3
assert abs(r2.json()['probability_disease'] - p[1]) < 1e-3
print('API and direct model agree')

2026-10-20 14:38:46,383 INFO model loaded from models\model.joblib (threshold=0.26)


2026-10-20 14:38:46,383 INFO GET /health status=200 latency_ms=2.2


{'status': 'ok', 'model_loaded': True}
2026-10-20 14:38:46,399 INFO POST /predict status=200 latency_ms=11.5


2026-10-20 14:38:46,416 INFO POST /predict status=200 latency_ms=10.7


2026-10-20 14:38:46,416 INFO POST /predict status=422 latency_ms=2.4


200 {'prediction': 1, 'label': 'heart disease', 'probability_predicted_class': 0.848, 'probability_disease': 0.848}
200 {'prediction': 0, 'label': 'no heart disease', 'probability_predicted_class': 0.9772, 'probability_disease': 0.0228}
422 Input should be less than or equal to 100
API and direct model agree


## Batch scoring of the cleaned dataset

The final cell applies the same saved pipeline to the full cleaned dataset to illustrate batch inference. These predictions are included as a demonstration of how the bundle can be reused; they are not a new evaluation set and should not be used to claim independent model performance.


In [5]:
df = pd.read_csv('data/heart_cleaned.csv')
pp = bundle['pipeline'].predict_proba(df[bundle['features']])[:, 1]
df['pred'] = (pp >= bundle['threshold']).astype(int)
print(pd.crosstab(df['target'], df['pred'], rownames=['actual'], colnames=['predicted']))
print('note: includes training rows, so this is not a generalisation estimate')

predicted    0    1
actual             
0          109   55
1            6  133
note: includes training rows, so this is not a generalisation estimate
